# 友安杯数据质量与特征分析

## 摘要

- `mask_x=False` 对应的 99 个数值特征全部为零，因此这些位置是占位数据，不能作为有效训练样本。
- 监督训练的有效样本条件应写为 `mask_x & mask_y & finite(y)`，即同时具备有效特征、属于标签范围且标签不是空值。
- Y1 是 `[0, 1]` 范围内的截面排名标签，均值约为 `0.5`，无需再次归一化，也不需要把靠近 0 或 1 的值当作异常值删除。
- 建模范围内的 99 个数值特征没有 `NaN` 或无穷值，训练期已经近似零均值、单位标准差；后续应重点处理厚尾和时间漂移。
- 类别特征大部分为低基数变量，可以使用稀疏独热编码（One-Hot）；高基数特征 `cat_5` 需要单独比较多种编码方案。

本 Notebook 是一份可重复执行的数据诊断与处理方案说明，不训练最终模型。阅读时建议先看四阶段切分和掩码定义，再依次查看标签、数值特征和类别特征分析。

## 分析背景与方法

分析的最小单位是一个 `(time, stock)` 位置，即某个时间点上的某只股票。所有统计都严格保留官方时间顺序，训练集、验证集和测试集不随机打乱，避免未来信息泄漏。

### 数据口径

- 预训练期：位于训练起点之前，没有可用于监督学习的真实标签。
- 训练期：用于拟合模型和所有数据处理参数。
- 验证期：用于时间外验证、选择模型和调整方案。
- 测试期：标签为空，只能生成预测，不能参与任何有监督统计。

### 关键假设

- `mask_x=True` 表示当前位置具有有效特征；`mask_x=False` 即使底层数组中存有数值，也必须先检查它是否只是占位。
- `mask_y=True` 表示该位置属于标签或评价股票池；训练和验证还必须检查标签是否为有限数。
- 数值特征的计数、均值、标准差、极值和非有限值率采用全量分块扫描。
- 分位数、四分位距异常率和群体稳定性指标（Population Stability Index，PSI）采用固定随机种子的时间分层抽样，兼顾代表性和内存占用。
- 训练期单特征秩相关系数（Rank Information Coefficient，RankIC）均匀抽取时间点，验证期使用全部时间点。

### 复现与内存约束

数据规模接近两千万个 `(time, stock)` 位置。本 Notebook 沿时间轴分块处理，并固定随机种子，避免一次性展开成超大型 Pandas 表，同时保证重复运行得到相同结果。

### 0. 环境与显示配置

导入数据处理和绘图库，并统一 Pandas 的数字显示格式、Matplotlib 的画布风格和中文字体。这里同时定义中文表头映射：内部字段仍保持英文，只在展示结果时翻译。

In [ ]:
# 导入运行本 Notebook 所需的标准库和第三方库。
# 统一显示配置，保证表格精度、图表风格和中文字体在各单元保持一致。

# 标准库
import pickle
from pathlib import Path

# 第三方库
import numpy as np
import pandas as pd
import zstandard as zstd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.20,
        "font.size": 10,
        "font.family": "sans-serif",
        "font.sans-serif": ["Microsoft YaHei", "SimHei", "DejaVu Sans"],
        "axes.unicode_minus": False,
    }
)

COLORS = {
    "blue": "#2F6BFF",
    "orange": "#E98B2A",
    "gold": "#D5A72E",
    "olive": "#758B36",
    "pink": "#C95C86",
    "ink": "#263238",
    "grey": "#8A949E",
}

# 面向读者的中文表头映射；内部 DataFrame 字段保持英文，避免影响计算逻辑。
DISPLAY_COLUMN_NAMES = {
    "key": "数据键",
    "count": "计数",
    "python_type": "Python 类型",
    "shape": "形状",
    "dtype": "数据类型",
    "split": "数据阶段",
    "start_inclusive": "起始索引（含）",
    "stop_exclusive": "结束索引（不含）",
    "time_count": "时间点数量",
    "mask_x_true_rate": "有效特征比例",
    "mask_x_per_time_median": "每时点有效特征股票数中位数",
    "mask_y_true_rate": "标签股票池比例",
    "finite_y1_rate": "Y1 有限标签比例",
    "usable_y1_rate": "Y1 可训练比例",
    "invalid_rows": "mask_x=False 行数",
    "numeric_zero_rate": "占位数值为零比例",
    "allzero_row_rate": "整行特征全零比例",
    "invalid_numeric_min": "占位数值最小值",
    "invalid_numeric_max": "占位数值最大值",
    "target": "预测目标",
    "n": "样本数量",
    "mean": "均值",
    "std": "标准差",
    "min": "最小值",
    "p001": "0.1% 分位数",
    "p01": "1% 分位数",
    "p05": "5% 分位数",
    "p25": "25% 分位数",
    "median": "中位数",
    "p75": "75% 分位数",
    "p95": "95% 分位数",
    "p99": "99% 分位数",
    "p999": "99.9% 分位数",
    "max": "最大值",
    "iqr_outlier_rate": "IQR 异常率",
    "rows": "有效样本行数",
    "sample_rows": "抽样行数",
    "feature": "特征",
    "feature_index": "特征序号",
    "train_mean": "训练期均值",
    "train_std": "训练期标准差",
    "nonfinite_rate": "非有限值比例",
    "zero_rate": "零值比例",
    "train_min": "训练期最小值",
    "train_p01": "训练期 1% 分位数",
    "train_median": "训练期中位数",
    "train_p99": "训练期 99% 分位数",
    "train_max": "训练期最大值",
    "max_abs": "最大绝对值",
    "train_outlier_rate": "训练期 IQR 异常率",
    "valid_outlier_rate": "验证期 IQR 异常率",
    "test_outlier_rate": "测试期 IQR 异常率",
    "valid_mean_shift_sd": "验证期均值漂移（标准差）",
    "test_mean_shift_sd": "测试期均值漂移（标准差）",
    "valid_psi": "验证期 PSI",
    "test_psi": "测试期 PSI",
    "max_abs_mean_shift": "最大绝对均值漂移",
    "max_psi": "最大 PSI",
    "valid_feature_rows": "有效特征行数",
    "duplicate_rows_beyond_first": "首行之外的重复行数",
    "duplicate_groups": "重复组数",
    "time_points_with_duplicates": "出现重复的时间点数",
    "duplicate_rate": "重复行比例",
    "mean_rank_ic": "平均 RankIC",
    "std_rank_ic": "RankIC 标准差",
    "positive_rate": "RankIC 为正比例",
    "abs_mean_rank_ic": "绝对平均 RankIC",
    "category_feature": "类别特征",
    "min_code": "最小编码",
    "max_code": "最大编码",
    "cardinality": "类别基数",
    "top1_share": "最大类别占比",
    "top5_share": "前五类别占比",
    "rare_share_count_lt100": "低频类别占比（计数<100）",
    "unseen_vs_train_rate": "相对训练期未见类别率",
    "change_rate": "相邻时间变化率",
    "valid_pairs": "有效相邻样本对数",
    "metric": "指标",
    "value": "数值",
    "feature_group": "特征组",
    "profile": "特征概况",
    "recommended_encoding": "推荐编码",
    "important_setting": "关键设置",
}

# 只翻译面向读者的分类值；技术变量名、特征名和数据类型保持原样。
DISPLAY_VALUE_NAMES = {
    "pretrain": "预训练期",
    "train": "训练期",
    "valid": "验证期",
    "test": "测试期",
    "train_sample": "训练期抽样",
    "Train rows": "训练样本行数",
    "One-Hot columns": "One-Hot 列数",
    "Non-zero values": "非零元素数量",
    "Estimated categorical CSR memory (MB)": "类别 CSR 估算内存（MB）",
    "Low-cardinality categorical features": "低基数类别特征",
    "High-cardinality, identifier-like feature": "高基数、近似身份标识的特征",
    "Already standardized; some heavy tails and drift": "已近似标准化，部分特征存在厚尾和漂移",
    "Sparse One-Hot": "稀疏 One-Hot",
    "Compare sparse One-Hot / native categorical / embedding": "比较稀疏 One-Hot、原生类别处理和 Embedding",
    "Raw baseline + rank/clip experiments": "原始值基线，并比较排名或截尾实验",
    "handle_unknown='ignore'": "设置 handle_unknown='ignore'",
    "Keep an explicit unknown bucket; avoid dense matrices": "保留未知类别桶，禁止使用密集矩阵",
    "Fit any thresholds on Train only": "所有阈值只在训练期拟合",
}

SPLIT_DISPLAY_NAMES = {
    "pretrain": "预训练期",
    "train": "训练期",
    "valid": "验证期",
    "test": "测试期",
}


def translate_display_label(label):
    """翻译表格的字段名、索引名和阶段名称，技术标识保持原样。"""
    if label is None:
        return None
    return DISPLAY_COLUMN_NAMES.get(
        label,
        DISPLAY_VALUE_NAMES.get(label, label),
    )


def translate_display_axis(axis):
    """同时支持普通索引和多级索引（MultiIndex）。"""
    if isinstance(axis, pd.MultiIndex):
        return pd.MultiIndex.from_tuples(
            [
                tuple(translate_display_label(part) for part in item)
                for item in axis
            ],
            names=[translate_display_label(name) for name in axis.names],
        )
    return pd.Index(
        [translate_display_label(item) for item in axis],
        name=translate_display_label(axis.name),
    )


def translate_display_label(label):
    """翻译表格的字段名、索引名和阶段名称，技术标识保持原样。"""
    if label is None:
        return None
    return DISPLAY_COLUMN_NAMES.get(
        label,
        DISPLAY_VALUE_NAMES.get(label, label),
    )


def translate_display_axis(axis):
    """同时支持普通索引和多级索引（MultiIndex）。"""
    if isinstance(axis, pd.MultiIndex):
        return pd.MultiIndex.from_tuples(
            [
                tuple(translate_display_label(part) for part in item)
                for item in axis
            ],
            names=[translate_display_label(name) for name in axis.names],
        )
    return pd.Index(
        [translate_display_label(item) for item in axis],
        name=translate_display_label(axis.name),
    )


def translate_display_label(label):
    """翻译表格的字段名、索引名和阶段名称，技术标识保持原样。"""
    if label is None:
        return None
    return DISPLAY_COLUMN_NAMES.get(
        label,
        DISPLAY_VALUE_NAMES.get(label, label),
    )


def translate_display_axis(axis):
    """同时支持普通索引和多级索引（MultiIndex）。"""
    if isinstance(axis, pd.MultiIndex):
        return pd.MultiIndex.from_tuples(
            [
                tuple(translate_display_label(part) for part in item)
                for item in axis
            ],
            names=[translate_display_label(name) for name in axis.names],
        )
    return pd.Index(
        [translate_display_label(item) for item in axis],
        name=translate_display_label(axis.name),
    )


def display_cn(value):
    """以中文展示表头、索引和分类值，不修改后续计算使用的原始对象。"""
    if isinstance(value, pd.DataFrame):
        translated = value.copy()
        translated.columns = translate_display_axis(translated.columns)
        translated.index = translate_display_axis(translated.index)
        translated = translated.replace(DISPLAY_VALUE_NAMES)
        return display(translated)
    if isinstance(value, pd.Series):
        translated = value.copy()
        translated.name = translate_display_label(translated.name)
        translated.index = translate_display_axis(translated.index)
        return display(translated.replace(DISPLAY_VALUE_NAMES))
    return display(value)


### 0.1 可复现参数

集中设置输入路径、随机种子、分块大小、抽样规模和排行榜数量。固定随机种子保证抽样统计和图表可以复现；分块参数控制峰值内存。

In [ ]:
# 集中管理路径、随机种子、分块大小、抽样规模和排行榜长度。
# 修改实验规模时应优先调整这些参数，不要在后续函数中散落硬编码数值。

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "01_analysis").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录。")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data.z"
OUTPUT_DIR = PROJECT_ROOT / "01_analysis" / "outputs"

ANALYSIS_SEED = 20260724
TIME_CHUNK_SIZE = 16
MAX_SAMPLE_TIME_POINTS = 300
SAMPLE_STOCKS_PER_TIME = 300
TRAIN_IC_TIME_POINTS = 80
TOP_N = 15

rng = np.random.default_rng(ANALYSIS_SEED)

## 数据与环境

### 1. 读取并检查源数据

**目的：**确认输入文件、数组形状、数据类型和官方切分边界，为后续所有统计建立统一口径。

`data.z` 是 Pickle 对象经过 Zstandard 压缩后的文件。读取时先取得压缩字节，再解压并反序列化。该步骤会把完整数据载入内存，因此后续分析只创建必要的分块或视图，避免无意义的数据复制。

In [ ]:
# 读取 Zstandard 压缩的 Pickle 数据：先读取压缩字节，再解压并反序列化。
# 返回值是包含特征、标签、掩码和时间边界的字典。

def read_zstd_pickle(path):
    compressed_bytes = pd.read_pickle(path)
    decompressed_bytes = zstd.ZstdDecompressor().decompress(compressed_bytes)
    return pickle.loads(decompressed_bytes)


data = read_zstd_pickle(DATA_PATH)
print(f"读取完成：{DATA_PATH.resolve()}")

### 1.1 数据结构检查

列出 `data.z` 中每个对象的 Python 类型、形状和数据类型。重点确认数值特征、类别特征、标签和掩码在前两个维度完全对齐。

In [ ]:
# 遍历数据字典，记录每个对象的类型、形状和数据类型。
# 该表只用于结构核对，不展开大型数组。

structure_rows = []
for key in sorted(data.keys()):
    value = data[key]
    structure_rows.append(
        {
            "key": key,
            "python_type": type(value).__name__,
            "shape": getattr(value, "shape", None),
            "dtype": str(getattr(value, "dtype", "")),
        }
    )

data_structure_df = pd.DataFrame(structure_rows)
display_cn(data_structure_df)

### 1.2 官方时间切分

读取训练、验证和测试起点，构造左闭右开的时间切片 `[start, stop)`。断言用于尽早发现边界顺序或数组形状错误。

In [ ]:
# 从数值特征形状读取时间长度、股票数量和数值特征数量。
# 使用断言验证官方边界顺序以及各数组前两维是否一致。

T, STOCK_COUNT, FEATURE_COUNT = data["num_x"].shape
CATEGORY_COUNT = data["cat_x"].shape[2]

train_start_idx = int(data["train_start_idx"])
valid_start_idx = int(data["valid_start_idx"])
test_start_idx = int(data["test_start_idx"])

assert 0 <= train_start_idx <= valid_start_idx <= test_start_idx <= T
assert data["cat_x"].shape[:2] == (T, STOCK_COUNT)
assert data["y1"].shape == (T, STOCK_COUNT)

split_slices = {
    "pretrain": slice(0, train_start_idx),
    "train": slice(train_start_idx, valid_start_idx),
    "valid": slice(valid_start_idx, test_start_idx),
    "test": slice(test_start_idx, T),
}

split_summary_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "start_inclusive": split_slice.start,
            "stop_exclusive": split_slice.stop,
            "time_count": split_slice.stop - split_slice.start,
        }
        for split_name, split_slice in split_slices.items()
    ]
)
display_cn(split_summary_df)

### 1.3 创建分阶段视图

为训练、验证和测试阶段建立数组切片。NumPy 基本切片通常返回视图，因此不会复制完整数据；后续仍需避免对这些视图进行原地修改。

In [ ]:
# 按官方时间边界建立训练、验证和测试视图。
# 这些切片用于后续分析，避免主动复制完整数组。

time_series_keys = ["num_x", "cat_x", "y1", "mask_x", "mask_y"]

train_data = {
    key: data[key][split_slices["train"]]
    for key in time_series_keys
}
valid_data = {
    key: data[key][split_slices["valid"]]
    for key in time_series_keys
}
test_data = {
    key: data[key][split_slices["test"]]
    for key in time_series_keys
}

print("训练集、验证集和测试集已按时间切分；切片是视图，不复制完整数组。")

## 分析结果

## 1. 掩码与股票池覆盖

本节先回答“哪些位置真正可以用于训练”，再观察四个时间阶段中股票池规模的变化。

四个核心概念如下：

- `mask_x=True`：该股票在该时间点具有有效特征。
- `mask_y=True`：该位置属于标签或评价股票池。
- `finite_y=True`：标签数组中实际存在有限答案，不是 `NaN` 或无穷值。
- `usable_y=True`：同时满足有效特征、标签范围和有限答案，可以用于监督训练。

最终训练条件是 `mask_x & mask_y & finite(y)`。不能仅凭特征数组里“有数值”就绕过 `mask_x`。

### 2.1 汇总掩码和标签覆盖率

按四个阶段统计有效特征、标签范围、有限标签和最终可用标签的比例，并计算每个时间点的股票数量。该表用于验证训练条件，也为后面的时间图提供数据。

In [ ]:
# 计算全时间轴上的掩码数量、有限标签数量和最终可用标签数量。
# 再按四个阶段汇总比例，检查训练口径是否一致。

mask_x = data["mask_x"]
mask_y = data["mask_y"]

# 每个时间点有效特征股票的数量。
mask_x_count = mask_x.sum(axis=1, dtype=np.int64)
mask_y_count = mask_y.sum(axis=1, dtype=np.int64)
finite_y1_count = np.isfinite(data["y1"]).sum(axis=1, dtype=np.int64)
# 最终监督样本必须同时满足有效特征、标签股票池和有限标签三个条件。
usable_y1_count = (
    mask_x & mask_y & np.isfinite(data["y1"])
).sum(axis=1, dtype=np.int64)

# 分阶段汇总比例，便于比较四个时间区间。
coverage_rows = []
for split_name, split_slice in split_slices.items():
    start, stop = split_slice.start, split_slice.stop
    total_positions = (stop - start) * STOCK_COUNT
    coverage_rows.append(
        {
            "split": split_name,
            "time_count": stop - start,
            "mask_x_true_rate": mask_x_count[start:stop].sum() / total_positions,
            "mask_x_per_time_median": np.median(mask_x_count[start:stop]),
            "mask_y_true_rate": mask_y_count[start:stop].sum() / total_positions,
            "finite_y1_rate": finite_y1_count[start:stop].sum() / total_positions,
            "usable_y1_rate": usable_y1_count[start:stop].sum() / total_positions,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
display_cn(coverage_df)

### 2.2 查看全时间轴覆盖变化

用背景色标出预训练、训练、验证和测试四个区间。上图比较特征股票池与标签股票池，下图比较实际存在的标签与可用于训练的标签。

In [ ]:
# 定义四阶段背景色和统一的时间分区标注函数。
# 主图比较特征股票池、标签股票池以及实际可用标签随时间的变化。

SPLIT_BAND_COLORS = {
    "pretrain": "#D8DEE6",
    "train": "#DCE8FF",
    "valid": "#FDE7CD",
    "test": "#E4EDD0",
}


# 背景色只用于辅助阅读，不参与任何计算。
def add_split_bands(axis, show_labels=False):
    for split_name, split_slice in split_slices.items():
        axis.axvspan(
            split_slice.start,
            split_slice.stop,
            color=SPLIT_BAND_COLORS[split_name],
            alpha=0.38,
            linewidth=0,
        )
        if show_labels:
            midpoint = (split_slice.start + split_slice.stop) / 2
            axis.text(
                midpoint,
                0.98,
                SPLIT_DISPLAY_NAMES[split_name],
                transform=axis.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=9,
                color=COLORS["ink"],
                bbox={"facecolor": "white", "alpha": 0.55, "edgecolor": "none", "pad": 1.5},
            )
    for boundary in (train_start_idx, valid_start_idx, test_start_idx):
        axis.axvline(boundary, color=COLORS["ink"], linewidth=0.9, alpha=0.65)


time_axis = np.arange(T)
# 上下两张图共享时间轴，阶段边界能够严格对齐。
figure, axes = plt.subplots(
    2,
    1,
    figsize=(13, 7.5),
    sharex=True,
    constrained_layout=True,
)

add_split_bands(axes[0], show_labels=True)
axes[0].plot(time_axis, mask_x_count, label="有效特征（mask_x=True）", color=COLORS["blue"], linewidth=1.4)
axes[0].plot(time_axis, mask_y_count, label="标签范围（mask_y=True）", color=COLORS["orange"], linewidth=1.2)
axes[0].set_title("特征与标签股票池覆盖随时间变化")
axes[0].set_ylabel("每个时间点的股票数量")
axes[0].legend(frameon=False, ncol=2, loc="lower right")

add_split_bands(axes[1])
axes[1].plot(time_axis, finite_y1_count, label="Y1 有限标签", color=COLORS["gold"], linewidth=1.2)
axes[1].plot(time_axis, usable_y1_count, label="Y1 可训练标签", color=COLORS["olive"], linewidth=1.0, alpha=0.85)
axes[1].set_title("实际存在与可用于训练的标签数量")
axes[1].set_xlabel("时间索引")
axes[1].set_ylabel("每个时间点的股票数量")
axes[1].legend(frameon=False, ncol=2, loc="upper left")

plt.show()

### 2.3 分阶段查看股票池

将四个阶段拆成小图，避免较短的验证期在全时间轴图中被压缩。重点观察股票池是否突然跳变、收缩或出现异常缺口。

In [ ]:
# 将四个阶段分别绘制，避免短区间在完整时间轴中难以观察。
# 每个子图使用相同纵轴，方便直接比较股票池规模。

figure, axes = plt.subplots(2, 2, figsize=(13, 7), sharey=True, constrained_layout=True)

for axis, (split_name, split_slice) in zip(axes.flat, split_slices.items()):
    local_time = np.arange(split_slice.start, split_slice.stop)
    axis.plot(
        local_time,
        mask_x_count[split_slice],
        label="有效特征（mask_x=True）",
        color=COLORS["blue"],
        linewidth=1.3,
    )
    axis.plot(
        local_time,
        mask_y_count[split_slice],
        label="标签范围（mask_y=True）",
        color=COLORS["orange"],
        linewidth=1.1,
    )
    axis.set_title(f"{SPLIT_DISPLAY_NAMES[split_name]}：[{split_slice.start}, {split_slice.stop})")
    axis.set_xlabel("时间索引")
    axis.set_ylabel("股票数量")

axes[0, 0].legend(frameon=False, loc="upper left")
figure.suptitle("按官方时间切分的股票池覆盖明细", fontsize=14, fontweight="bold")
plt.show()

### 2.4 验证 `mask_x=False` 是否只是占位

全量分块扫描无效特征位置，分别计算单个数值为零的比例和整行 99 个数值全部为零的比例。断言要求四个阶段的结果都为 100%。

In [ ]:
# 分块扫描所有 `mask_x=False` 的位置，避免一次性复制海量占位数据。
# 分别统计单值为零和整行特征全零的比例。

def summarize_mask_placeholders(split_name, split_slice, chunk_size=TIME_CHUNK_SIZE):
    invalid_rows = 0
    invalid_zero_elements = 0
    invalid_elements = 0
    invalid_allzero_rows = 0
    invalid_min = np.inf
    invalid_max = -np.inf

    # 沿时间维分块，避免一次性取出所有无效位置。
    for chunk_start in range(split_slice.start, split_slice.stop, chunk_size):
        chunk_stop = min(chunk_start + chunk_size, split_slice.stop)
        invalid_mask = ~mask_x[chunk_start:chunk_stop]
        if not np.any(invalid_mask):
            continue
        values = data["num_x"][chunk_start:chunk_stop][invalid_mask]
        invalid_rows += values.shape[0]
        invalid_elements += values.size
        invalid_zero_elements += np.count_nonzero(values == 0)
        invalid_allzero_rows += np.count_nonzero(np.all(values == 0, axis=1))
        invalid_min = min(invalid_min, float(values.min()))
        invalid_max = max(invalid_max, float(values.max()))

    return {
        "split": split_name,
        "invalid_rows": invalid_rows,
        "numeric_zero_rate": invalid_zero_elements / max(invalid_elements, 1),
        "allzero_row_rate": invalid_allzero_rows / max(invalid_rows, 1),
        "invalid_numeric_min": invalid_min,
        "invalid_numeric_max": invalid_max,
    }


placeholder_df = pd.DataFrame(
    [
        summarize_mask_placeholders(split_name, split_slice)
        for split_name, split_slice in split_slices.items()
    ]
)
display_cn(placeholder_df)

# 若断言失败，说明 `mask_x=False` 不能再简单解释为全零占位。
assert np.allclose(placeholder_df["numeric_zero_rate"], 1.0)
assert np.allclose(placeholder_df["allzero_row_rate"], 1.0)
print("结论：mask_x=False 的数值特征全部为全零占位，不进入监督训练。")

### 掩码结论

全量检查显示，`mask_x=False` 位置上的数值特征全部为零，属于明确的占位数据。因此：

1. `mask_x=False` 的位置不能进入监督训练，也不应参与特征分布统计。
2. 训练集和验证集使用 `mask_x & mask_y & finite(y)` 作为有效样本条件。
3. 测试集的 Y1 为空是正常现象，表示比赛没有提供答案；预测时只在官方要求的股票池上生成结果。
4. 如果后续模型需要固定形状张量，可以保留零占位用于对齐，但损失函数和评价指标必须通过掩码排除这些位置。

## 2. 标签分析

**目的：**判断 Y1 的数值范围、分布形态、时间稳定性以及是否需要归一化或异常值处理。

本节分别计算训练期和验证期的样本量、均值、标准差、多个分位数和四分位距（IQR）异常率；随后绘制直方图，并按时间截面观察均值和标准差。

由于比赛评价关注排序能力，标签是否已经是排名形式会直接影响后续损失函数和预处理选择。

### 3.1 标签数值概况

只在有效监督样本上统计 Y1。多组分位数用于确认值域和尾部；IQR 异常率用于判断是否存在超出常规分布范围的值。

In [ ]:
# 在训练期和验证期的有效监督样本上汇总 Y1。
# 分位数描述标签范围，IQR 规则用于检查是否存在异常尾部。

LABEL_QUANTILES = [0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1]
label_profile_rows = []
label_samples = {}

for target_name in ("y1",):
    target = data[target_name]
    for split_name in ("train", "valid"):
        split_slice = split_slices[split_name]
        usable_mask = (
            mask_x[split_slice]
            & mask_y[split_slice]
            & np.isfinite(target[split_slice])
        )
        values = target[split_slice][usable_mask].astype(np.float64, copy=False)
        quantiles = np.quantile(values, LABEL_QUANTILES)
        # 用当前阶段自身的四分位数描述标签中部和尾部。
        q1, median, q3 = quantiles[4], quantiles[5], quantiles[6]
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        label_profile_rows.append(
            {
                "target": target_name,
                "split": split_name,
                "n": values.size,
                "mean": values.mean(),
                "std": values.std(),
                "min": quantiles[0],
                "p001": quantiles[1],
                "p01": quantiles[2],
                "p05": quantiles[3],
                "p25": q1,
                "median": median,
                "p75": q3,
                "p95": quantiles[7],
                "p99": quantiles[8],
                "p999": quantiles[9],
                "max": quantiles[10],
                "iqr_outlier_rate": np.mean((values < lower) | (values > upper)),
            }
        )

        sample_size = min(values.size, 200_000)
        sampled_indices = rng.choice(values.size, size=sample_size, replace=False)
        label_samples[(target_name, split_name)] = values[sampled_indices]

label_profile_df = pd.DataFrame(label_profile_rows)
display_cn(label_profile_df)

### 3.2 标签分布图

训练集和验证集分别抽样绘制直方图。所有子图使用相同坐标范围，便于直接比较两个阶段之间的分布差异。

In [ ]:
# 使用固定随机种子抽样标签并绘制直方图。
# 两个子图共享坐标尺度，确保训练期和验证期可以公平比较。

figure, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharex=True, sharey=True, constrained_layout=True)

for column_index, split_name in enumerate(("train", "valid")):
    axis = axes[column_index]
    axis.hist(
        label_samples[("y1", split_name)],
        bins=50,
        range=(0, 1),
        color=COLORS["blue"] if split_name == "train" else COLORS["orange"],
        alpha=0.82,
        edgecolor="white",
        linewidth=0.25,
    )
    axis.axvline(0.5, color=COLORS["ink"], linewidth=1, linestyle="--")
    axis.set_title(f"Y1—{SPLIT_DISPLAY_NAMES[split_name]}")
    axis.set_xlabel("标签值")
    axis.set_ylabel("样本数量")

figure.suptitle("Y1 标签分布", fontsize=14, fontweight="bold")
plt.show()


### 3.3 标签的时间稳定性

逐时间截面计算 Y1 标签的均值和标准差，用于检查排名标签是否稳定。

In [ ]:
# 逐时间截面计算 Y1 标签的均值和标准差。
# 只使用同时满足特征、标签股票池和有限标签条件的样本。

label_time_rows = []

# 按时间截面计算，不能把所有时间点混在一起。
for time_idx in range(train_start_idx, test_start_idx):
    target = data["y1"]
    usable_mask = (
        mask_x[time_idx]
        & mask_y[time_idx]
        & np.isfinite(target[time_idx])
    )
    values = target[time_idx, usable_mask]
    if values.size:
        q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
        label_time_rows.append(
            {
                "time": time_idx,
                "target": "y1",
                "n": values.size,
                "mean": values.mean(),
                "std": values.std(),
                "q1": q1,
                "median": median,
                "q3": q3,
            }
        )

label_time_df = pd.DataFrame(label_time_rows)

figure, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, constrained_layout=True)
for axis in axes:
    add_split_bands(axis)

axes[0].plot(label_time_df["time"], label_time_df["mean"], label="Y1", color=COLORS["blue"], linewidth=1)
axes[1].plot(label_time_df["time"], label_time_df["std"], label="Y1", color=COLORS["blue"], linewidth=1)

axes[0].set_title("Y1 标签截面均值")
axes[1].set_title("Y1 标签截面标准差")
axes[1].set_xlabel("时间索引")
for axis in axes:
    axis.legend(frameon=False, ncol=2)

plt.show()


### 标签处理结论

Y1 已经是 `[0, 1]` 范围内的截面排名标签：均值约为 `0.5`，标准差约为 `0.289`，训练期和验证期分布高度一致。

- 不需要再次使用标准化或最小最大归一化，否则只会重复变换。
- 靠近 0 和 1 的值表示截面排名两端，不是错误数据，因此不应作为极端值删除。
- 模型输出无需被强制限制在 `[0, 1]`，只要保持正确的相对排序即可。

## 3. 数值特征质量与时间漂移

**目的：**检查 99 个数值特征是否存在空值、无穷值、常数列、极端值、重复行和跨阶段分布变化，并用单特征 RankIC 初步识别排序信号。

### 统计口径

- 训练、验证和测试阶段统一使用 `mask_x & mask_y`，与建模或评价股票池保持一致。
- 预训练期没有真实标签，因此不进入有监督特征统计。
- 计数、均值、标准差、零值率、极值和非有限值率使用全量分块扫描。
- 分位数、IQR 异常率和 PSI 使用固定随机种子的时间分层抽样。
- 所有截尾阈值、分箱边界和转换参数只能由训练期拟合，不能读取验证期或测试期信息。

### 指标解释

- **IQR 异常率：**落在 `[Q1-1.5×IQR, Q3+1.5×IQR]` 之外的样本占比，用于识别厚尾，不代表这些样本一定错误。
- **均值漂移：**验证或测试均值相对训练均值偏移了多少个训练标准差。
- **PSI：**比较两个阶段的分布差异；数值越大，说明时间漂移越明显，但仍需结合模型验证决定是否删除特征。

### 4.1 全量扫描与分层抽样

每个阶段先分块累计精确统计量，再按时间分层抽取有限样本估计分位数。这样既保留全量质量检查，又避免构造数百万行的临时表。

In [ ]:
# 对每个阶段执行全量分块扫描，并保留用于分位数估计的时间分层样本。
# 精确统计与抽样统计分开计算，以控制内存同时保证质量检查完整。

def profile_numeric_split(split_name, split_slice, random_generator):
    count = 0
    sums = np.zeros(FEATURE_COUNT, dtype=np.float64)
    squared_sums = np.zeros(FEATURE_COUNT, dtype=np.float64)
    zero_counts = np.zeros(FEATURE_COUNT, dtype=np.int64)
    nonfinite_counts = np.zeros(FEATURE_COUNT, dtype=np.int64)
    minimum = np.full(FEATURE_COUNT, np.inf)
    maximum = np.full(FEATURE_COUNT, -np.inf)

    # 第一遍：全量累计精确统计量。
    for chunk_start in range(split_slice.start, split_slice.stop, TIME_CHUNK_SIZE):
        chunk_stop = min(chunk_start + TIME_CHUNK_SIZE, split_slice.stop)
        selected_mask = (
            mask_x[chunk_start:chunk_stop]
            & mask_y[chunk_start:chunk_stop]
        )
        values = data["num_x"][chunk_start:chunk_stop][selected_mask]
        if values.size == 0:
            continue
        count += values.shape[0]
        sums += values.sum(axis=0, dtype=np.float64)
        squared_sums += np.einsum("ij,ij->j", values, values, dtype=np.float64)
        zero_counts += np.count_nonzero(values == 0, axis=0)
        nonfinite_counts += np.count_nonzero(~np.isfinite(values), axis=0)
        minimum = np.minimum(minimum, values.min(axis=0))
        maximum = np.maximum(maximum, values.max(axis=0))

    mean = sums / max(count, 1)
    std = np.sqrt(np.maximum(squared_sums / max(count, 1) - mean**2, 0))

    sampled_rows = []
    # 第二步：沿时间轴均匀取点，再在每个时间点抽取股票。
    sampled_times = np.linspace(
        split_slice.start,
        split_slice.stop - 1,
        min(split_slice.stop - split_slice.start, MAX_SAMPLE_TIME_POINTS),
        dtype=int,
    )
    for time_idx in sampled_times:
        available_stocks = np.flatnonzero(mask_x[time_idx] & mask_y[time_idx])
        if available_stocks.size == 0:
            continue
        sample_size = min(SAMPLE_STOCKS_PER_TIME, available_stocks.size)
        chosen_stocks = random_generator.choice(
            available_stocks,
            size=sample_size,
            replace=False,
        )
        sampled_rows.append(data["num_x"][time_idx, chosen_stocks])

    sample = np.concatenate(sampled_rows, axis=0)
    quantiles = np.quantile(sample, [0.01, 0.25, 0.5, 0.75, 0.99], axis=0)

    return {
        "split": split_name,
        "n": count,
        "mean": mean,
        "std": std,
        "zero_rate": zero_counts / max(count, 1),
        "nonfinite_rate": nonfinite_counts / max(count, 1),
        "min": minimum,
        "max": maximum,
        "p01": quantiles[0],
        "p25": quantiles[1],
        "median": quantiles[2],
        "p75": quantiles[3],
        "p99": quantiles[4],
        "sample": sample,
    }


numeric_profiles = {
    split_name: profile_numeric_split(split_name, split_slices[split_name], rng)
    for split_name in ("train", "valid", "test")
}

sampling_summary_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": profile["n"],
            "sample_rows": profile["sample"].shape[0],
        }
        for split_name, profile in numeric_profiles.items()
    ]
)

display_cn(sampling_summary_df)

### 4.2 计算异常率、漂移和 PSI

IQR 阈值与 PSI 分箱边界全部由训练期样本确定，再原样应用到验证和测试阶段。排行榜表保留内部英文列名用于计算，展示时自动转换成中文表头。

In [ ]:
# 使用训练期分位数定义 IQR 阈值和 PSI 分箱，再应用到验证与测试阶段。
# 由训练期拟合阈值可以避免把未来分布信息泄漏到数据处理流程。

# PSI 的分箱边界只由训练期样本确定。
def calculate_psi(train_values, comparison_values, feature_index):
    bin_edges = np.unique(
        np.quantile(train_values[:, feature_index], np.linspace(0, 1, 11))
    )
    if bin_edges.size < 3:
        return 0.0
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    train_counts = np.histogram(
        train_values[:, feature_index],
        bins=bin_edges,
    )[0].astype(np.float64)
    comparison_counts = np.histogram(
        comparison_values[:, feature_index],
        bins=bin_edges,
    )[0].astype(np.float64)

    train_share = (train_counts + 0.5) / (train_counts.sum() + 0.5 * train_counts.size)
    comparison_share = (comparison_counts + 0.5) / (
        comparison_counts.sum() + 0.5 * comparison_counts.size
    )
    return float(
        np.sum(
            (comparison_share - train_share)
            * np.log(comparison_share / train_share)
        )
    )


train_profile = numeric_profiles["train"]
# IQR 阈值也只由训练期确定，防止未来信息泄漏。
train_iqr = train_profile["p75"] - train_profile["p25"]
outlier_lower = train_profile["p25"] - 1.5 * train_iqr
outlier_upper = train_profile["p75"] + 1.5 * train_iqr

outlier_rates = {
    split_name: np.mean(
        (profile["sample"] < outlier_lower)
        | (profile["sample"] > outlier_upper),
        axis=0,
    )
    for split_name, profile in numeric_profiles.items()
}

valid_psi = np.asarray(
    [
        calculate_psi(
            train_profile["sample"],
            numeric_profiles["valid"]["sample"],
            feature_index,
        )
        for feature_index in range(FEATURE_COUNT)
    ]
)
test_psi = np.asarray(
    [
        calculate_psi(
            train_profile["sample"],
            numeric_profiles["test"]["sample"],
            feature_index,
        )
        for feature_index in range(FEATURE_COUNT)
    ]
)

numeric_feature_rows = []
for feature_index in range(FEATURE_COUNT):
    valid_shift = (
        numeric_profiles["valid"]["mean"][feature_index]
        - train_profile["mean"][feature_index]
    ) / max(train_profile["std"][feature_index], 1e-12)
    test_shift = (
        numeric_profiles["test"]["mean"][feature_index]
        - train_profile["mean"][feature_index]
    ) / max(train_profile["std"][feature_index], 1e-12)

    numeric_feature_rows.append(
        {
            "feature": f"num_{feature_index}",
            "feature_index": feature_index,
            "train_mean": train_profile["mean"][feature_index],
            "train_std": train_profile["std"][feature_index],
            "nonfinite_rate": train_profile["nonfinite_rate"][feature_index],
            "zero_rate": train_profile["zero_rate"][feature_index],
            "train_min": train_profile["min"][feature_index],
            "train_p01": train_profile["p01"][feature_index],
            "train_median": train_profile["median"][feature_index],
            "train_p99": train_profile["p99"][feature_index],
            "train_max": train_profile["max"][feature_index],
            "max_abs": max(
                abs(train_profile["min"][feature_index]),
                abs(train_profile["max"][feature_index]),
            ),
            "train_outlier_rate": outlier_rates["train"][feature_index],
            "valid_outlier_rate": outlier_rates["valid"][feature_index],
            "test_outlier_rate": outlier_rates["test"][feature_index],
            "valid_mean_shift_sd": valid_shift,
            "test_mean_shift_sd": test_shift,
            "valid_psi": valid_psi[feature_index],
            "test_psi": test_psi[feature_index],
        }
    )

numeric_feature_df = pd.DataFrame(numeric_feature_rows)
numeric_feature_df["max_abs_mean_shift"] = numeric_feature_df[
    ["valid_mean_shift_sd", "test_mean_shift_sd"]
].abs().max(axis=1)
numeric_feature_df["max_psi"] = numeric_feature_df[
    ["valid_psi", "test_psi"]
].max(axis=1)

display_cn(
    numeric_feature_df.sort_values(
        ["max_abs_mean_shift", "max_psi"],
        ascending=False,
    ).head(20)
)

assert numeric_feature_df["nonfinite_rate"].max() == 0
assert (numeric_feature_df["train_std"] > 0).all()

### 4.3 数值特征排行榜

四张水平条形图分别展示最大绝对值、训练期 IQR 异常率、训练至验证均值漂移和训练至测试 PSI。排行榜只用于定位需要优先检查的特征。

In [ ]:
# 将数值特征质量指标绘制为四个 Top-N 水平排行榜。
# 排序只决定展示顺序，不修改底层特征表。

def horizontal_leaderboard(axis, frame, value_column, title, color, absolute=False):
    selected = frame.copy()
    selected["_sort_value"] = (
        selected[value_column].abs() if absolute else selected[value_column]
    )
    selected = selected.nlargest(TOP_N, "_sort_value").sort_values("_sort_value")
    values = selected[value_column]
    bar_colors = color
    if absolute:
        bar_colors = [
            COLORS["blue"] if value >= 0 else COLORS["orange"]
            for value in values
        ]
    axis.barh(selected["feature"], values, color=bar_colors, alpha=0.88)
    axis.set_title(title)
    axis.set_xlabel(DISPLAY_COLUMN_NAMES.get(value_column, value_column))
    if values.min() < 0 < values.max():
        axis.axvline(0, color=COLORS["ink"], linewidth=0.8)


figure, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
horizontal_leaderboard(
    axes[0, 0],
    numeric_feature_df,
    "max_abs",
    "绝对值最大的数值特征",
    COLORS["blue"],
)
horizontal_leaderboard(
    axes[0, 1],
    numeric_feature_df,
    "train_outlier_rate",
    "训练期 IQR 异常率最高的特征",
    COLORS["gold"],
)
horizontal_leaderboard(
    axes[1, 0],
    numeric_feature_df,
    "valid_mean_shift_sd",
    "训练期至验证期均值漂移最大的特征",
    COLORS["blue"],
    absolute=True,
)
horizontal_leaderboard(
    axes[1, 1],
    numeric_feature_df,
    "test_psi",
    "训练期至测试期 PSI 最大的特征",
    COLORS["olive"],
)
plt.show()

### 4.4 完全重复特征行检查

重复值在每个时间截面内检查：先用数值特征的字节表示快速分组，再用类别特征确认整行是否完全一致。这样可避免把不同时间的正常重复状态误判为重复样本。

In [ ]:
# 在每个时间截面内检查数值特征与类别特征完全一致的重复行。
# 两阶段分组先缩小候选集合，再精确确认，降低比较成本。

def rows_as_byte_keys(rows):
    contiguous_rows = np.ascontiguousarray(rows)
    row_width = contiguous_rows.dtype.itemsize * contiguous_rows.shape[1]
    return contiguous_rows.view(np.dtype((np.void, row_width))).reshape(-1)


# 重复定义：同一时间截面内，99 个数值特征和 9 个类别特征全部一致。
def count_exact_duplicate_feature_rows(split_slice):
    valid_rows = 0
    duplicate_rows = 0
    duplicate_groups = 0
    time_points_with_duplicates = 0

    for time_idx in range(split_slice.start, split_slice.stop):
        valid_stocks = mask_x[time_idx]
        numeric_rows = data["num_x"][time_idx, valid_stocks]
        category_rows = data["cat_x"][time_idx, valid_stocks]
        valid_rows += numeric_rows.shape[0]
        if numeric_rows.shape[0] < 2:
            continue

        # 先按数值行的原始字节快速寻找候选重复组。
        numeric_keys = rows_as_byte_keys(numeric_rows)
        _, group_ids, counts = np.unique(
            numeric_keys,
            return_inverse=True,
            return_counts=True,
        )

        duplicates_this_time = 0
        groups_this_time = 0
        for group_id in np.flatnonzero(counts > 1):
            candidate_categories = category_rows[group_ids == group_id]
            category_keys = rows_as_byte_keys(candidate_categories)
            _, full_counts = np.unique(category_keys, return_counts=True)
            repeated_counts = full_counts[full_counts > 1]
            duplicates_this_time += int(np.sum(repeated_counts - 1))
            groups_this_time += int(repeated_counts.size)

        if duplicates_this_time:
            time_points_with_duplicates += 1
            duplicate_rows += duplicates_this_time
            duplicate_groups += groups_this_time

    return {
        "valid_feature_rows": valid_rows,
        "duplicate_rows_beyond_first": duplicate_rows,
        "duplicate_groups": duplicate_groups,
        "time_points_with_duplicates": time_points_with_duplicates,
        "duplicate_rate": duplicate_rows / max(valid_rows, 1),
    }


duplicate_rows = []
for split_name, split_slice in split_slices.items():
    result = count_exact_duplicate_feature_rows(split_slice)
    result["split"] = split_name
    duplicate_rows.append(result)

duplicate_df = pd.DataFrame(duplicate_rows)
display_cn(duplicate_df)

### 4.5 单特征 RankIC 计算

对每个时间截面分别把特征值和标签转换为秩，再计算皮尔逊相关系数，即得到 RankIC。训练期均匀抽取 80 个时间点，验证期使用全部时间点。

In [ ]:
# 将截面特征和标签分别转成秩，并逐列计算秩相关系数（RankIC）。
# 对无有效方差的特征返回缺失值，汇总时通过有效计数安全处理。

def rank_columns(values):
    return pd.DataFrame(values).rank(method="average", axis=0).to_numpy(
        dtype=np.float64,
        copy=True,
    )


def calculate_univariate_rank_ic(target, time_indices):
    time_ic_rows = []
    for time_idx in time_indices:
        usable_mask = (
            mask_x[time_idx]
            & mask_y[time_idx]
            & np.isfinite(target[time_idx])
        )
        if np.count_nonzero(usable_mask) < 20:
            continue

        features = data["num_x"][time_idx, usable_mask]
        labels = target[time_idx, usable_mask]
        # 评价指标关注排序，因此先把原始数值转换为截面秩。
        feature_ranks = rank_columns(features)
        label_ranks = pd.Series(labels).rank(method="average").to_numpy(
            dtype=np.float64,
            copy=True,
        )

        feature_ranks -= feature_ranks.mean(axis=0)
        label_ranks -= label_ranks.mean()
        denominator = np.sqrt(
            np.einsum("ij,ij->j", feature_ranks, feature_ranks)
            * np.dot(label_ranks, label_ranks)
        )
        correlations = np.divide(
            feature_ranks.T @ label_ranks,
            denominator,
            out=np.full(FEATURE_COUNT, np.nan),
            where=denominator > 0,
        )
        time_ic_rows.append(correlations)

    return np.vstack(time_ic_rows)


train_ic_times = np.linspace(
    train_start_idx,
    valid_start_idx - 1,
    TRAIN_IC_TIME_POINTS,
    dtype=int,
)
valid_ic_times = np.arange(valid_start_idx, test_start_idx)

ic_summary_rows = []
for target_name in ("y1",):
    for split_name, time_indices in (
        ("train_sample", train_ic_times),
        ("valid", valid_ic_times),
    ):
        ic_matrix = calculate_univariate_rank_ic(data[target_name], time_indices)
        # 使用显式有效计数，避免全空特征触发均值或标准差警告。
        finite_ic = np.isfinite(ic_matrix)
        valid_count = finite_ic.sum(axis=0)
        ic_sum = np.where(finite_ic, ic_matrix, 0).sum(axis=0)
        ic_squared_sum = np.where(finite_ic, ic_matrix**2, 0).sum(axis=0)
        mean_ic = np.divide(
            ic_sum,
            valid_count,
            out=np.full(FEATURE_COUNT, np.nan),
            where=valid_count > 0,
        )
        variance_ic = np.divide(
            ic_squared_sum,
            valid_count,
            out=np.full(FEATURE_COUNT, np.nan),
            where=valid_count > 0,
        ) - mean_ic**2
        std_ic = np.sqrt(np.maximum(variance_ic, 0))
        positive_rate = np.divide(
            ((ic_matrix > 0) & finite_ic).sum(axis=0),
            valid_count,
            out=np.full(FEATURE_COUNT, np.nan),
            where=valid_count > 0,
        )
        for feature_index in range(FEATURE_COUNT):
            ic_summary_rows.append(
                {
                    "target": target_name,
                    "split": split_name,
                    "feature": f"num_{feature_index}",
                    "feature_index": feature_index,
                    "mean_rank_ic": mean_ic[feature_index],
                    "std_rank_ic": std_ic[feature_index],
                    "positive_rate": positive_rate[feature_index],
                }
            )

ic_summary_df = pd.DataFrame(ic_summary_rows)

for target_name in ("y1",):
    print(f"验证集 {target_name.upper()} 单特征 RankIC")
    display_cn(
        ic_summary_df[
            (ic_summary_df["target"] == target_name)
            & (ic_summary_df["split"] == "valid")
        ]
        .assign(abs_mean_rank_ic=lambda frame: frame["mean_rank_ic"].abs())
        .sort_values("abs_mean_rank_ic", ascending=False)
        .head(20)
    )

### 4.6 RankIC 稳定性对比

按验证期绝对平均 RankIC 选择候选特征，并同时绘制训练期抽样结果。若方向和大小在两个阶段明显不一致，应警惕过拟合或时间漂移。

In [ ]:
# 选择验证期绝对平均 RankIC 较高的特征。
# 同图比较训练期抽样与验证期结果，用于观察信号方向是否稳定。

target_ic = ic_summary_df[ic_summary_df["target"] == "y1"]
valid_ic = target_ic[target_ic["split"] == "valid"].copy()
valid_ic["abs_ic"] = valid_ic["mean_rank_ic"].abs()
top_features = valid_ic.nlargest(TOP_N, "abs_ic")["feature"].tolist()

comparison = (
    target_ic[target_ic["feature"].isin(top_features)]
    .pivot(index="feature", columns="split", values="mean_rank_ic")
    .loc[top_features]
    .sort_values("valid")
)

figure, axis = plt.subplots(figsize=(7, 6), constrained_layout=True)
y_positions = np.arange(comparison.shape[0])
axis.barh(
    y_positions - 0.18,
    comparison["train_sample"],
    height=0.34,
    label="训练期抽样",
    color=COLORS["grey"],
)
axis.barh(
    y_positions + 0.18,
    comparison["valid"],
    height=0.34,
    label="验证集",
    color=COLORS["blue"],
)
axis.set_yticks(y_positions, comparison.index)
axis.axvline(0, color=COLORS["ink"], linewidth=0.8)
axis.set_title("Y1 单特征 RankIC")
axis.set_xlabel("平均 RankIC")
axis.legend(frameon=False)

plt.show()


### 数值特征处理结论

- 建模范围内没有 `NaN` 或无穷值，所有训练期特征标准差均大于零。
- 数值特征在训练期已经接近零均值、单位标准差，不建议重复使用普通 `StandardScaler`。
- 极端值不直接删除；后续应比较原始值、训练期分位数截尾和逐时间截面排名三种方案。
- 均值漂移或 PSI 较高只表示分布变化明显，不等于特征无效，应通过走步验证观察其 RankIC 和模型贡献是否稳定。
- 单特征 RankIC 排行榜用于发现候选信号，不等同于多特征模型中的最终重要性。

## 4. 类别特征分析与编码建议

**目的：**了解每个类别特征的基数、类别集中度、时间变化率以及验证集和测试集中未见类别的比例，从而选择适合模型和内存规模的编码方式。

主要指标如下：

- **基数：**训练期出现过的不同类别数量。
- **头部集中度：**出现最多的 1 个或 5 个类别所占比例。
- **未见类别率：**验证集或测试集中未在训练期出现的类别占比。
- **相邻时间变化率：**同一股票的类别值在相邻时间点发生变化的比例，用于判断特征更像稳定身份还是动态状态。

独热编码必须使用稀疏矩阵；如果转换成密集矩阵，会造成不可接受的内存占用。

### 5.1 类别基数与未见类别

先扫描每个类别特征的编码范围，再按阶段累计频数。训练集的类别集合是唯一基准，验证和测试中的新编码都计入未见类别率。

In [ ]:
# 全量扫描类别编码范围，并按四个阶段累计类别出现次数。
# 训练期类别集合用来计算验证和测试中的未见类别率。

category_min = np.full(CATEGORY_COUNT, np.iinfo(np.int64).max, dtype=np.int64)
category_max = np.full(CATEGORY_COUNT, np.iinfo(np.int64).min, dtype=np.int64)

for chunk_start in range(0, T, 32):
    chunk_stop = min(chunk_start + 32, T)
    values = data["cat_x"][chunk_start:chunk_stop][mask_x[chunk_start:chunk_stop]]
    if values.size:
        category_min = np.minimum(category_min, values.min(axis=0))
        category_max = np.maximum(category_max, values.max(axis=0))

category_ranges = category_max - category_min + 1
assert np.all(category_ranges < 2_000_000)

# 分阶段累计频数，后续以训练期零计数位置识别未见类别。
category_counts = {}
for split_name, split_slice in split_slices.items():
    counts_by_feature = [
        np.zeros(int(category_ranges[feature_index]), dtype=np.int64)
        for feature_index in range(CATEGORY_COUNT)
    ]
    row_count = 0

    for chunk_start in range(split_slice.start, split_slice.stop, 32):
        chunk_stop = min(chunk_start + 32, split_slice.stop)
        selected_mask = mask_x[chunk_start:chunk_stop]
        if split_name != "pretrain":
            selected_mask &= mask_y[chunk_start:chunk_stop]
        values = data["cat_x"][chunk_start:chunk_stop][selected_mask]
        if values.size == 0:
            continue
        row_count += values.shape[0]
        for feature_index in range(CATEGORY_COUNT):
            counts_by_feature[feature_index] += np.bincount(
                values[:, feature_index] - category_min[feature_index],
                minlength=int(category_ranges[feature_index]),
            )

    category_counts[split_name] = {
        "row_count": row_count,
        "counts": counts_by_feature,
    }

category_profile_rows = []
for feature_index in range(CATEGORY_COUNT):
    train_counts = category_counts["train"]["counts"][feature_index]
    for split_name in split_slices:
        split_counts = category_counts[split_name]["counts"][feature_index]
        row_count = category_counts[split_name]["row_count"]
        nonzero_counts = split_counts[split_counts > 0]
        unseen_count = (
            split_counts[train_counts == 0].sum()
            if split_name != "train"
            else 0
        )
        category_profile_rows.append(
            {
                "category_feature": f"cat_{feature_index}",
                "feature_index": feature_index,
                "split": split_name,
                "rows": row_count,
                "min_code": category_min[feature_index],
                "max_code": category_max[feature_index],
                "cardinality": nonzero_counts.size,
                "top1_share": nonzero_counts.max() / max(row_count, 1),
                "top5_share": np.sort(nonzero_counts)[-5:].sum() / max(row_count, 1),
                "rare_share_count_lt100": (
                    nonzero_counts[nonzero_counts < 100].sum() / max(row_count, 1)
                ),
                "unseen_vs_train_rate": unseen_count / max(row_count, 1),
            }
        )

category_profile_df = pd.DataFrame(category_profile_rows)
display_cn(category_profile_df)

### 5.2 类别的相邻时间变化率

仅比较相邻两个时间点都满足 `mask_x=True` 的同一只股票。变化率低的特征更像稳定身份，变化率高的特征更像随时间更新的状态。

In [ ]:
# 计算同一股票在相邻有效时间点之间的类别变化率。
# 仅比较前后两个时间点都具有有效特征的位置。

category_change_rows = []
for split_name, split_slice in split_slices.items():
    changed_counts = np.zeros(CATEGORY_COUNT, dtype=np.int64)
    eligible_pairs = 0

    for time_idx in range(max(split_slice.start + 1, 1), split_slice.stop):
        # 只有相邻两个时间点都有效的位置才进入变化率分母。
        eligible_mask = mask_x[time_idx] & mask_x[time_idx - 1]
        eligible_count = np.count_nonzero(eligible_mask)
        eligible_pairs += eligible_count
        if eligible_count:
            changed_counts += np.count_nonzero(
                data["cat_x"][time_idx, eligible_mask]
                != data["cat_x"][time_idx - 1, eligible_mask],
                axis=0,
            )

    for feature_index in range(CATEGORY_COUNT):
        category_change_rows.append(
            {
                "category_feature": f"cat_{feature_index}",
                "split": split_name,
                "change_rate": changed_counts[feature_index] / max(eligible_pairs, 1),
            }
        )

category_change_df = pd.DataFrame(category_change_rows)

category_overview_df = (
    category_profile_df[
        category_profile_df["split"].isin(["train", "valid", "test"])
    ]
    .pivot(
        index="category_feature",
        columns="split",
        values=["cardinality", "unseen_vs_train_rate", "top1_share"],
    )
)
display_cn(category_overview_df)
display_cn(
    category_change_df.pivot(
        index="category_feature",
        columns="split",
        values="change_rate",
    )
)

### 5.3 类别分布与 One-Hot 内存估算

左图展示训练期类别基数，右图展示验证和测试相对训练期的未见类别率。随后按 CSR 稀疏矩阵结构估算独热编码的非零元素和内存占用。

In [ ]:
# 展示训练期类别基数和验证、测试未见类别率。
# 根据每行恰有 9 个非零值的假设估算 CSR 稀疏 One-Hot 内存。

train_category_profile = category_profile_df[
    category_profile_df["split"] == "train"
].sort_values("feature_index")
valid_category_profile = category_profile_df[
    category_profile_df["split"] == "valid"
].sort_values("feature_index")
test_category_profile = category_profile_df[
    category_profile_df["split"] == "test"
].sort_values("feature_index")

figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].barh(
    train_category_profile["category_feature"],
    train_category_profile["cardinality"],
    color=COLORS["blue"],
)
axes[0].set_xscale("log")
axes[0].set_title("训练期类别基数")
axes[0].set_xlabel("类别基数（对数刻度）")

y_positions = np.arange(CATEGORY_COUNT)
axes[1].barh(
    y_positions - 0.18,
    valid_category_profile["unseen_vs_train_rate"],
    height=0.34,
    label="验证集",
    color=COLORS["orange"],
)
axes[1].barh(
    y_positions + 0.18,
    test_category_profile["unseen_vs_train_rate"],
    height=0.34,
    label="测试集",
    color=COLORS["olive"],
)
axes[1].set_yticks(y_positions, train_category_profile["category_feature"])
axes[1].set_title("相对训练期的未见类别率")
axes[1].set_xlabel("样本占比")
axes[1].legend(frameon=False)

plt.show()

total_onehot_columns = int(train_category_profile["cardinality"].sum())
train_rows = int(category_counts["train"]["row_count"])
# One-Hot 后每个原始类别字段恰好贡献一个非零元素。
estimated_nonzero_values = train_rows * CATEGORY_COUNT
estimated_csr_mb = (
    estimated_nonzero_values * (4 + 4)
    + (train_rows + 1) * 4
) / 1024**2

onehot_feasibility_df = pd.DataFrame(
    {
        "metric": [
            "Train rows",
            "One-Hot columns",
            "Non-zero values",
            "Estimated categorical CSR memory (MB)",
        ],
        "value": [
            train_rows,
            total_onehot_columns,
            estimated_nonzero_values,
            estimated_csr_mb,
        ],
    }
)
display_cn(onehot_feasibility_df)

### 5.4 编码方案

根据基数和内存规模给出分组建议。低基数类别直接使用稀疏 One-Hot；高基数 `cat_5` 需要与原生类别模型或 Embedding 做时间验证对比。

In [ ]:
# 汇总类别和数值特征的推荐编码方案。
# 该表是后续建模实验的起点，不直接改变当前数据。

encoding_plan_df = pd.DataFrame(
    [
        {
            "feature_group": "cat_0,1,2,3,4,6,7,8",
            "profile": "Low-cardinality categorical features",
            "recommended_encoding": "Sparse One-Hot",
            "important_setting": "handle_unknown='ignore'",
        },
        {
            "feature_group": "cat_5",
            "profile": "High-cardinality, identifier-like feature",
            "recommended_encoding": "Compare sparse One-Hot / native categorical / embedding",
            "important_setting": "Keep an explicit unknown bucket; avoid dense matrices",
        },
        {
            "feature_group": "num_x",
            "profile": "Already standardized; some heavy tails and drift",
            "recommended_encoding": "Raw baseline + rank/clip experiments",
            "important_setting": "Fit any thresholds on Train only",
        },
    ]
)
display_cn(encoding_plan_df)

## 主要结论与下一步

1. **有效样本：**监督训练严格使用 `mask_x & mask_y & finite(y)`；`mask_x=False` 的数值是全零占位。
2. **标签：**Y1 已是 `[0, 1]` 截面排名标签，不再归一化，也不删除排名两端的值。
3. **数值特征：**不重复使用普通标准化；重点比较原始值、逐时间截面排名和训练期分位数截尾。
4. **时间漂移：**持续监控均值漂移、PSI 和滚动 RankIC；高漂移特征是否保留由时间验证结果决定。
5. **类别特征：**低基数类别使用稀疏独热编码；高基数 `cat_5` 单独比较稀疏 One-Hot、原生类别模型和嵌入（Embedding）。
6. **内存：**继续沿时间轴分块，不把接近两千万个 `(time, stock)` 位置一次性展开成大型 Pandas DataFrame。

建议保持官方时间切分不变，依次进行以下对照实验：

```text
A. 原始数值特征
B. 数值特征 + 逐时间截面排名
C. 数值特征 + 训练期分位数截尾
D. 数值特征 + 低基数类别的稀疏 One-Hot
E. 高基数类别的原生类别模型或 Embedding
```

每次实验只改变一个处理因素，并同时记录训练期、验证期的 RankIC、稳定性和内存占用，避免无法判断性能变化来自哪一步。